# Notebook 13 — Appendix B Charts

**Pipeline position:** runs after `10_Robustness_Regression.ipynb` (needs regression outputs)  
**Inputs:** `intermediary/Master.csv`, `intermediary/NaturalResource.csv`, `intermediary/clusters_k5_agg.csv`  
**Outputs:** `Visualisation/charts/appendix/`  

Charts produced for Appendix B:
- Chart B1: Cluster world map (k=5, 1995)
- Chart B2: ML feature importance consensus
- Chart B3: Train vs Test R²
- Chart B4: Coefficient forest plot (Models 3a–3d)
- Chart B5: Human Capital × NR Production interaction

# Appendix B: Specification Sensitivity Charts

**Capstone Project — Moody's Ratings**
*Industrial Upgrading in Emerging Markets with Rich Natural Resources*

Five charts supporting Appendix B of the report. Each chart is saved as
HTML (interactive) and PNG (static) to `Visualisation/charts/appendix/`.

| # | Chart | Source |
|---|-------|--------|
| B1 | Cluster world map (k=5, 1995) | NB4 pipeline |
| B2 | ML feature importance consensus | NB5 pipeline |
| B3 | ML train vs test R² | NB5 pipeline |
| B4 | Regression coefficient forest (3a–3d) | NB6 pipeline |
| B5 | HCI × NR Production interaction | NB6 pipeline |


## 0. Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, silhouette_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Project root ─────────────────────────────────────────────────────────────
def _find_root(marker='intermediary'):
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    fallback = '/Users/leoss/Desktop/GitHub/Capstone/CLEAN'
    if os.path.isdir(os.path.join(fallback, marker)):
        return fallback
    raise RuntimeError("Could not find project root")

ROOT = _find_root()
os.chdir(ROOT)

OUT = os.path.join(os.path.expanduser('~'), 'Downloads', 'capstone_charts', 'appendix')
os.makedirs(OUT, exist_ok=True)

# ── Style constants (shared across all charts) ──────────────────────────────
FONT = 'Public Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
CFG  = {'displayModeBar': False, 'responsive': True}

PAL = {'blue': '#4a6fa5', 'red': '#c23a3a', 'green': '#2e7d4a',
       'gold': '#c9a227', 'grey': '#999999', 'orange': '#d4853b'}

CLUSTER_COLORS = {
    'Petrostates':             '#E63946',
    'Oil Exporters':           '#457B9D',
    'Diversified Producers':   '#2A9D8F',
    'Low Intensity Producers': '#E9C46A',
    'Hard Mineral Exporters':  '#A8DADC',
    'Mineral-Rich Developing': '#A8DADC',
    'Low-Intensity Producers': '#E9C46A',
}

SPEC_COLORS = {
    '3a': '#4a6fa5', '3b': '#c23a3a', '3c': '#2e7d4a', '3d': '#d4853b',
}

def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=60, r=40, t=50, b=60))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_html(f"{path}.html", config=CFG)
    print(f"  Saved: {path}.html")
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=3)
        print(f"  Saved: {path}.png")
    except Exception:
        print(f"  (PNG skipped - install kaleido)")

print(f'Root: {ROOT}')
print(f'Output: {OUT}')


Root: /Users/leoss/Desktop/GitHub/Capstone/CLEAN
Output: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix


## 1. Load Data

In [2]:
ECI_COL = 'Economic Complexity Index'

master = pd.read_csv('intermediary/Master.csv', dtype={'Country Code': str})
nr     = pd.read_csv('intermediary/NaturalResource.csv')

# 47-country sample
include_list = [
    'AGO','ALB','ARG','AZE','BHR','BIH','BOL','BWA','CHL',
    'CMR','COD','COG','COL','DOM','DZA','ECU','EGY','EST',
    'GAB','GHA','GIN','IDN','IRN','IRQ','JAM','KAZ','KWT',
    'MEX','MMR','MNG','MRT','MYS','NGA','OMN','PAK','PER',
    'QAT','ROU','RUS','SAU','TUN','UKR','VEN','VNM','YEM',
    'ZAF','ZWE',
]

# Cluster assignments (k=5, aggregated)
clusters = pd.read_csv('intermediary/clusters_k5_agg.csv', dtype={'Country Code': str})
cl_map   = clusters[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')

# Master panel filtered to sample
df = master[(master['Year'].between(1995, 2019)) &
            (master['Country Code'].isin(include_list))].copy()
df = df.merge(cl_map, on='Country Code', how='inner')
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs')
print(f'Years: {df["Year"].min():.0f}-{df["Year"].max():.0f}')


Sample: 47 countries, 1,175 obs
Years: 1995-2019


## Chart B1 — Cluster World Map (k=5, 1995)

Countries coloured by their k=5 cluster assignment from the 1995 snapshot.
Countries producing more than 15% of a resource's global output are outlined
with a dark border.


In [3]:
nr_sample = nr[nr['Country Code'].isin(include_list)]
cl_1995 = clusters[['Country Code', 'Country', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')

# Compute dominance (>15% of global production for any resource)
df_total = nr_sample[nr_sample['Year'] == 1995].pivot_table(
    index=['Country', 'Country Code'],
    columns='Resource', values='Production_TotalValue', aggfunc='sum',
).reset_index().fillna(0)

prod_cols = [c for c in df_total.columns if c not in ['Country', 'Country Code']]
for col in prod_cols:
    total = df_total[col].sum()
    if total > 0:
        df_total[f'{col}_Share'] = (df_total[col] / total) * 100

share_cols = [c for c in df_total.columns if c.endswith('_Share')]
df_map = cl_1995.merge(df_total[['Country Code'] + share_cols], on='Country Code', how='left')
df_map['Is_Dominant'] = (df_map[share_cols].fillna(0) >= 15.0).any(axis=1)

cluster_names = dict(zip(cl_1995['Cluster'], cl_1995['ClusterLabels']))

fig_b1 = go.Figure()
for cid in sorted(df_map['Cluster'].unique()):
    lbl   = cluster_names.get(cid, f'Cluster {cid}')
    color = CLUSTER_COLORS.get(lbl, '#aaa')

    sub = df_map[(df_map['Cluster'] == cid) & (~df_map['Is_Dominant'])]
    if len(sub) > 0:
        fig_b1.add_trace(go.Choropleth(
            locations=sub['Country Code'], z=[cid]*len(sub),
            colorscale=[[0, color], [1, color]], showscale=False,
            showlegend=True, name=lbl,
            hovertemplate='<b>' + sub['Country'].values + '</b><br>' + lbl + '<extra></extra>',
            marker=dict(line=dict(color='white', width=0.6)),
        ))

    sub_d = df_map[(df_map['Cluster'] == cid) & (df_map['Is_Dominant'])]
    if len(sub_d) > 0:
        fig_b1.add_trace(go.Choropleth(
            locations=sub_d['Country Code'], z=[cid]*len(sub_d),
            colorscale=[[0, color], [1, color]], showscale=False,
            showlegend=False,
            hovertemplate='<b>' + sub_d['Country'].values + '</b><br>' + lbl + ' (major producer)<extra></extra>',
            marker=dict(line=dict(color='#111', width=2.2)),
        ))

fig_b1.update_geos(
    showframe=False, showcoastlines=True, coastlinecolor='#ccc',
    showland=True, landcolor='#f0f0f0', showcountries=True, countrycolor='#ddd',
    projection_type='natural earth',
    lataxis_range=[-55, 80], lonaxis_range=[-170, 180],
)
fig_b1.update_layout(**base_layout(
    height=520, margin=dict(l=10, r=10, t=40, b=10),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.05, xanchor='center', x=0.5,
        font=dict(size=11), bgcolor='rgba(255,255,255,0.85)',
        bordercolor=GRID, borderwidth=1,
    ),
))
save(fig_b1, 'B1_cluster_map_k5_1995', w=1400, h=520)
fig_b1.show(config=CFG)


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B1_cluster_map_k5_1995.html


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B1_cluster_map_k5_1995.png


## 2. ML Data Preparation and Model Fitting

In [4]:
# ── Feature engineering (mirrors NB5) ─────────────────────────────────────────
ml = df.copy()
ml['Total_Production_Value_Per_Capita'] = (
    ml['Total_Production_Value'] / ml['Population'].replace(0, np.nan))
ml['L1_ECI']    = ml.groupby('Country Code')[ECI_COL].shift(1)
ml['ECI_delta'] = ml[ECI_COL] - ml['L1_ECI']
ml = ml.dropna(subset=['L1_ECI', ECI_COL, 'ECI_delta'])

log_cols = [
    'Human capital index',
    'Total_Production_Value_Per_Capita',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Government revenue',
    'Use of IMF credit (DOD, current US$)',
]
ml[log_cols] = np.log1p(ml[log_cols]).replace([np.inf, -np.inf], np.nan)

hci_m  = ml['Human capital index'].mean()
prod_m = ml['Total_Production_Value_Per_Capita'].mean()
gfcf_m = ml['Gross fixed capital formation, all, Constant prices, Percent of GDP'].mean()
ml['HCI_x_ProductionValue']  = (ml['Human capital index'] - hci_m) * \
                                (ml['Total_Production_Value_Per_Capita'] - prod_m)
ml['GFCF_x_ProductionValue'] = (ml['Gross fixed capital formation, all, Constant prices, Percent of GDP'] - gfcf_m) * \
                                (ml['Total_Production_Value_Per_Capita'] - prod_m)

base_features = [
    'Total_Production_Value_Per_Capita', 'Human capital index',
    'Rule of law index', 'Political stability \u2014 estimate',
    'Trade (% of GDP)',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Share of investment in GDP', 'Domestic credit to private sector (% of GDP)',
    'Landlocked', 'Urban population (% of total population)',
    'Government revenue', 'Capital depreciation rate',
    'Use of IMF credit (DOD, current US$)', 'Real interest rate (%)',
    'Inflation, consumer prices (annual %)', 'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)', 'L1_ECI',
]
all_features = base_features + ['HCI_x_ProductionValue', 'GFCF_x_ProductionValue']
ml = ml.dropna(subset=all_features)

name_map = {
    'Total_Production_Value_Per_Capita': 'Prod Value p.c.',
    'Human capital index': 'Human Capital',
    'Rule of law index': 'Rule of Law',
    'Political stability \u2014 estimate': 'Political Stability',
    'Trade (% of GDP)': 'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Capital Formation',
    'Share of investment in GDP': 'Investment Share',
    'Domestic credit to private sector (% of GDP)': 'Domestic Credit',
    'Landlocked': 'Landlocked',
    'Urban population (% of total population)': 'Urban Population',
    'Government revenue': 'Gov Revenue',
    'Capital depreciation rate': 'Depreciation',
    'Use of IMF credit (DOD, current US$)': 'IMF Credit',
    'Real interest rate (%)': 'Real Rate',
    'Inflation, consumer prices (annual %)': 'Inflation',
    'Access to electricity (% of population)': 'Electricity',
    'Adjusted savings: gross savings (% of GNI)': 'Gross Savings',
    'L1_ECI': 'Lagged ECI',
    'HCI_x_ProductionValue': 'HC \u00d7 Production',
    'GFCF_x_ProductionValue': 'GFCF \u00d7 Production',
}
short = [name_map.get(f, f) for f in all_features]
EXCLUDE_LABEL = 'Lagged ECI'

# ── PanelTemporalCV ──────────────────────────────────────────────────────────
class PanelTemporalCV:
    def __init__(self, years, n_splits=5, gap=1, min_train_years=8):
        self.years = np.asarray(years)
        uy = np.sort(np.unique(self.years))
        ec = uy[0] + min_train_years - 1
        lc = uy[-1] - gap - 1
        self.cutoffs = np.unique(np.linspace(ec, lc, n_splits).astype(int))
        self.n_splits = len(self.cutoffs)
        self.gap = gap
    def split(self, X=None, y=None, groups=None):
        for c in self.cutoffs:
            ti = np.where(self.years <= c)[0]
            vi = np.where(self.years > c + self.gap)[0]
            if len(ti) and len(vi): yield ti, vi
    def get_n_splits(self, X=None, y=None, groups=None): return self.n_splits

# ── Train / test split ───────────────────────────────────────────────────────
train_df = ml[ml['Year'] <= 2014].copy()
test_df  = ml[ml['Year'] >= 2015].copy()

y_tr_lv = train_df[ECI_COL].values
y_te_lv = test_df[ECI_COL].values
y_tr_dl = train_df['ECI_delta'].values
y_te_dl = test_df['ECI_delta'].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[all_features].values)
X_test  = scaler.transform(test_df[all_features].values)

tscv = PanelTemporalCV(train_df['Year'].values, n_splits=5, gap=1, min_train_years=8)

# ── Fit ECI level models ─────────────────────────────────────────────────────
print('Fitting ECI level models...')
lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_lv)
elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                 random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_lv)
models_lv = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic, 'Random Forest': rf}

# ── Fit delta-ECI models ─────────────────────────────────────────────────────
print('Fitting delta-ECI models...')
lasso_d   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
ridge_d   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_dl)
elastic_d = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
rf_d      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                   random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_dl)
models_dl = {'LASSO': lasso_d, 'Ridge': ridge_d, 'Elastic Net': elastic_d, 'Random Forest': rf_d}

# ── Performance tables ───────────────────────────────────────────────────────
def perf_table(models, y_tr, y_te, X_tr, X_te):
    rows = []
    for name, m in models.items():
        rows.append({
            'Model': name,
            'Train R\u00b2': round(r2_score(y_tr, m.predict(X_tr)), 4),
            'Test R\u00b2':  round(r2_score(y_te, m.predict(X_te)), 4),
        })
    return pd.DataFrame(rows)

perf_lv = perf_table(models_lv, y_tr_lv, y_te_lv, X_train, X_test)
perf_dl = perf_table(models_dl, y_tr_dl, y_te_dl, X_train, X_test)

# ── Feature importance (min-max normalised |coef|) ───────────────────────────
imp_df = pd.DataFrame({'Feature': all_features, 'Short': short})
for name, m in {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic}.items():
    c = np.abs(m.coef_)
    imp_df[name] = c / c.max() if c.max() > 0 else c
imp_df['RF'] = rf.feature_importances_ / rf.feature_importances_.max()
imp_df['Mean'] = imp_df[['LASSO', 'Ridge', 'Elastic Net']].mean(axis=1)
imp_df = imp_df[imp_df['Short'] != EXCLUDE_LABEL].sort_values('Mean', ascending=False)

print(f'ML sample: {ml["Country Code"].nunique()} countries, {len(ml):,} obs')
print(f'Train: {len(train_df):,}, Test: {len(test_df):,}')
print(f'Features: {len(all_features)}')
print()
print(perf_lv.to_string(index=False))


Fitting ECI level models...


Fitting delta-ECI models...


ML sample: 45 countries, 1,078 obs
Train: 853, Test: 225
Features: 20

        Model  Train R²  Test R²
        LASSO    0.8599   0.9183
        Ridge    0.8712   0.9141
  Elastic Net    0.8670   0.9141
Random Forest    0.8928   0.9292


## Chart B2 — ML Feature Importance Consensus (LASSO / Ridge / Elastic Net)

Normalised absolute coefficients (min-max scaled to 0-1). Lagged ECI excluded
because it is trivially the strongest predictor. Connecting lines show the
range of importance across the three regularised models.


In [5]:
d = imp_df.head(15).iloc[::-1].reset_index(drop=True)
lin_models = ['LASSO', 'Ridge', 'Elastic Net']
symbols = {'LASSO': 'circle', 'Ridge': 'square', 'Elastic Net': 'triangle-up'}
colors  = {'LASSO': PAL['red'], 'Ridge': PAL['blue'], 'Elastic Net': PAL['green']}

fig_b2 = go.Figure()

# Connecting lines
for _, row in d.iterrows():
    vals = [row[m] for m in lin_models if not np.isnan(row[m])]
    if len(vals) >= 2:
        fig_b2.add_shape(type='line',
                       x0=min(vals), x1=max(vals), y0=row['Short'], y1=row['Short'],
                       line=dict(color='#b0c0d4', width=2))

for m in lin_models:
    fig_b2.add_trace(go.Scatter(
        x=d[m], y=d['Short'], mode='markers', name=m,
        marker=dict(symbol=symbols[m], color=colors[m], size=11,
                    line=dict(width=1.2, color='white')),
        hovertemplate=f'%{{y}}: %{{x:.3f}}<extra>{m}</extra>',
    ))

fig_b2.update_layout(**base_layout(
    height=560, margin=dict(l=180, r=60, t=40, b=60),
    xaxis=dict(title='Normalised Feature Importance (min-max, 0\u20131)',
               gridcolor=GRID, gridwidth=0.5, range=[-0.02, 1.08]),
    yaxis=dict(tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5,
                font=dict(size=11)),
))
save(fig_b2, 'B2_ml_feature_importance_consensus', w=1200, h=560)
fig_b2.show(config=CFG)


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B2_ml_feature_importance_consensus.html


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B2_ml_feature_importance_consensus.png


## Chart B3 — Train vs Test R² (ECI Level and Delta-ECI)

Side-by-side comparison of in-sample and out-of-sample R-squared for LASSO,
Ridge, Elastic Net, and Random Forest. Left panel: ECI levels. Right panel:
year-on-year ECI changes. The large gap in the right panel reflects the
difficulty of predicting short-run variation in economic complexity.


In [6]:
fig_b3 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                       subplot_titles=['ECI Level', '\u0394ECI'])

for col_idx, perf in enumerate([perf_lv, perf_dl], 1):
    show = (col_idx == 1)
    fig_b3.add_trace(go.Bar(
        x=perf['Model'], y=perf['Train R\u00b2'], name='Train R\u00b2',
        marker_color=PAL['blue'], opacity=0.85, legendgroup='train',
        text=[f'{v:.3f}' for v in perf['Train R\u00b2']], textposition='outside',
        textfont=dict(size=10, color=PAL['blue']),
        showlegend=show,
    ), row=1, col=col_idx)
    fig_b3.add_trace(go.Bar(
        x=perf['Model'], y=perf['Test R\u00b2'], name='Test R\u00b2',
        marker_color=PAL['red'], opacity=0.85, legendgroup='test',
        text=[f'{v:.3f}' for v in perf['Test R\u00b2']], textposition='outside',
        textfont=dict(size=10, color=PAL['red']),
        showlegend=show,
    ), row=1, col=col_idx)
    fig_b3.update_xaxes(tickangle=-25, tickfont=dict(size=10), row=1, col=col_idx)
    fig_b3.update_yaxes(title_text='R\u00b2' if col_idx == 1 else '',
                        gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

fig_b3.update_layout(
    template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(family=FONT, size=11, color=NAVY),
    barmode='group', height=480, margin=dict(l=60, r=40, t=50, b=80),
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5,
                font=dict(size=12)),
)
save(fig_b3, 'B3_ml_train_vs_test_r2', w=1200, h=480)
fig_b3.show(config=CFG)


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B3_ml_train_vs_test_r2.html


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B3_ml_train_vs_test_r2.png


## 3. Regression Models (3a-3d)

In [7]:
# ── Feature engineering for regressions ────────────────────────────────────────
rdf = df.copy()
rdf['Total_Production_Value_Per_Capita'] = (
    rdf['Total_Production_Value'] / rdf['Population'].replace(0, np.nan))

rdf['log_HCI']              = np.log1p(rdf['Human capital index'].clip(lower=0))
rdf['log_GFCF']             = np.log1p(
    rdf['Gross fixed capital formation, all, Constant prices, Percent of GDP'].clip(lower=0))
rdf['log_Production_Value'] = np.log1p(rdf['Total_Production_Value_Per_Capita'].clip(lower=0))

rdf['ECI_lag1']  = rdf.groupby('Country Code')[ECI_COL].shift(1)
rdf['delta_ECI'] = rdf[ECI_COL] - rdf['ECI_lag1']

BASE_INDEP = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate',
    'Rule of law index',
    'log_Production_Value',
    'Trade (% of GDP)',
]
EXTRA_CONTROLS = [
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]

for var in BASE_INDEP:
    rdf[f'{var}_lag1'] = rdf.groupby('Country Code')[var].shift(1)

hci_c  = rdf['log_HCI']             - rdf['log_HCI'].mean()
gfcf_c = rdf['log_GFCF']            - rdf['log_GFCF'].mean()
prod_c = rdf['log_Production_Value'] - rdf['log_Production_Value'].mean()
rdf['log_HCI_x_log_Production']  = hci_c  * prod_c
rdf['log_GFCF_x_log_Production'] = gfcf_c * prod_c

hci_l_c  = rdf['log_HCI_lag1']             - rdf['log_HCI_lag1'].mean()
gfcf_l_c = rdf['log_GFCF_lag1']            - rdf['log_GFCF_lag1'].mean()
prod_l_c = rdf['log_Production_Value_lag1'] - rdf['log_Production_Value_lag1'].mean()
rdf['log_HCI_x_log_Production_lag1']  = hci_l_c  * prod_l_c
rdf['log_GFCF_x_log_Production_lag1'] = gfcf_l_c * prod_l_c

DISPLAY_LABELS = {
    'const':                                           'Constant',
    'log_HCI':                                         'Human Capital (log)',
    'log_GFCF':                                        'GFCF (log)',
    'Political stability \u2014 estimate':            'Political Stability',
    'Rule of law index':                               'Rule of Law',
    'log_Production_Value':                            'NR Production (log, pc)',
    'Trade (% of GDP)':                                'Trade (% GDP)',
    'log_HCI_x_log_Production':                        'HCI \u00d7 Production',
    'log_GFCF_x_log_Production':                       'GFCF \u00d7 Production',
    'ECI_lag1':                                        'ECI (t-1)',
    'log_HCI_lag1':                                    'Human Capital (t-1)',
    'log_GFCF_lag1':                                   'GFCF (t-1)',
    'Political stability \u2014 estimate_lag1':       'Political Stability (t-1)',
    'Rule of law index_lag1':                          'Rule of Law (t-1)',
    'log_Production_Value_lag1':                       'NR Production (t-1)',
    'Trade (% of GDP)_lag1':                           'Trade (t-1)',
    'log_HCI_x_log_Production_lag1':                   'HCI \u00d7 Prod (t-1)',
    'log_GFCF_x_log_Production_lag1':                  'GFCF \u00d7 Prod (t-1)',
    'Hydrocarbons_Dominant':                           'Hydrocarbons dom.',
    'Subsoil_Metals_Dominant':                         'Subsoil Metals dom.',
    'Precious_Metals_Dominant':                        'Precious Metals dom.',
    'Access to electricity (% of population)':         'Electricity access',
}

# ── OLS helper ────────────────────────────────────────────────────────────────
def run_ols(df_in, dv, regressors, cluster_by='Country Code'):
    req = [dv] + regressors + [cluster_by]
    sub = df_in.dropna(subset=req).copy()
    X   = sm.add_constant(sub[regressors])
    y   = sub[dv]
    res = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': sub[cluster_by]})
    return res, sub

def coef_table(res):
    tbl = pd.DataFrame({
        'Variable': res.params.index,
        'Coef':     res.params.values,
        'SE':       res.bse.values,
        'p':        res.pvalues.values,
        'CI_lo':    res.conf_int().iloc[:, 0].values,
        'CI_hi':    res.conf_int().iloc[:, 1].values,
    })
    tbl['Stars'] = tbl['p'].apply(
        lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else '')
    return tbl

REG_RESULTS = {}

# 3a: Base (no lag)
VARS_3A = BASE_INDEP + ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3a, sub_3a = run_ols(rdf, ECI_COL, VARS_3A)
REG_RESULTS['3a'] = (res_3a, coef_table(res_3a), sub_3a)
print(f'3a  N={int(res_3a.nobs):,}  R\u00b2={res_3a.rsquared:.4f}')

# 3b: Base + ECI lag
VARS_3B = BASE_INDEP + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3b, sub_3b = run_ols(rdf, ECI_COL, VARS_3B)
REG_RESULTS['3b'] = (res_3b, coef_table(res_3b), sub_3b)
print(f'3b  N={int(res_3b.nobs):,}  R\u00b2={res_3b.rsquared:.4f}')

# 3c: All regressors lagged
LAGGED_BASE = [f'{v}_lag1' for v in BASE_INDEP]
VARS_3C = LAGGED_BASE + ['ECI_lag1', 'log_HCI_x_log_Production_lag1', 'log_GFCF_x_log_Production_lag1']
res_3c, sub_3c = run_ols(rdf, ECI_COL, VARS_3C)
REG_RESULTS['3c'] = (res_3c, coef_table(res_3c), sub_3c)
print(f'3c  N={int(res_3c.nobs):,}  R\u00b2={res_3c.rsquared:.4f}')

# 3d: Extended controls + lag
VARS_3D = BASE_INDEP + EXTRA_CONTROLS + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3d, sub_3d = run_ols(rdf, ECI_COL, VARS_3D)
REG_RESULTS['3d'] = (res_3d, coef_table(res_3d), sub_3d)
print(f'3d  N={int(res_3d.nobs):,}  R\u00b2={res_3d.rsquared:.4f}')


3a  N=1,173  R²=0.4229
3b  N=1,126  R²=0.8773
3c  N=1,126  R²=0.8773
3d  N=1,120  R²=0.8800


## Chart B4 — Coefficient Forest Plot (Models 3a-3d)

Point estimates and 95% confidence intervals for the core regressors across
four specifications. Standard errors are clustered by country. Model 3e (first
differences) is excluded because its differenced regressors are not directly
comparable to the level coefficients shown here. Coefficients whose confidence
intervals cross zero are not statistically significant at the 5% level.


In [8]:
FOREST_VARS = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate', 'Rule of law index',
    'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]

MODEL_ORDER  = ['3a', '3b', '3c', '3d']
MODEL_LABELS = {
    '3a': '3a Base', '3b': '3b + Lag',
    '3c': '3c All Lagged', '3d': '3d Extended',
}

y_labels = [DISPLAY_LABELS.get(v, v) for v in FOREST_VARS]
y_pos    = {lbl: i for i, lbl in enumerate(y_labels)}
n_specs  = len(MODEL_ORDER)
offsets  = np.linspace(-0.22, 0.22, n_specs)

fig_b4 = go.Figure()
fig_b4.add_vline(x=0, line=dict(color='#aab0ba', width=1.5, dash='dash'))

for i in range(len(y_labels)):
    if i % 2 == 0:
        fig_b4.add_hrect(y0=i - 0.48, y1=i + 0.48,
                         fillcolor='rgba(230,235,242,0.35)', line=dict(width=0), layer='below')

for i, m in enumerate(MODEL_ORDER):
    if m not in REG_RESULTS: continue
    _, tbl, _ = REG_RESULTS[m]
    col = SPEC_COLORS[m]
    lbl = MODEL_LABELS[m]

    matched = tbl[tbl['Variable'].isin(FOREST_VARS)].copy()
    matched['Label'] = matched['Variable'].map(lambda v: DISPLAY_LABELS.get(v, v))
    matched['y_pos'] = matched['Label'].map(y_pos).fillna(-1) + offsets[i]
    matched = matched[matched['y_pos'] >= 0]

    fig_b4.add_trace(go.Scatter(
        x=matched['Coef'], y=matched['y_pos'],
        mode='markers',
        marker=dict(size=9, color=col, line=dict(color='white', width=1.5)),
        error_x=dict(
            type='data', symmetric=False,
            array=(matched['CI_hi'] - matched['Coef']).values,
            arrayminus=(matched['Coef'] - matched['CI_lo']).values,
            color=col, thickness=2.0, width=5,
        ),
        name=lbl,
        text=matched['Label'],
        hovertemplate='<b>%{text}</b><br>\u03b2 = %{x:.4f}<extra>' + lbl + '</extra>',
    ))

fig_b4.update_layout(**base_layout(
    height=max(560, len(y_labels) * 50 + 150),
    margin=dict(l=220, r=60, t=55, b=60),
    xaxis=dict(title='Coefficient (95% CI, SE clustered by country)',
               gridcolor=GRID, gridwidth=0.5, zeroline=False),
    yaxis=dict(
        tickvals=list(y_pos.values()),
        ticktext=list(y_pos.keys()),
        tickfont=dict(size=11), showgrid=True, gridcolor=GRID, gridwidth=0.5,
        range=[-0.55, len(y_labels) - 0.45],
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig_b4, 'B4_reg_coefficient_forest_3a_3d', w=1200, h=max(560, len(y_labels) * 50 + 150))
fig_b4.show(config=CFG)


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B4_reg_coefficient_forest_3a_3d.html


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B4_reg_coefficient_forest_3a_3d.png


## Chart B5 — Human Capital x NR Production Interaction

Scatter of log(HCI) against ECI, with observations coloured by per-capita
NR production value quartile. Dotted lines are OLS trendlines per quartile.
If the HCI x Production interaction were positive, the slope of HCI on ECI
would steepen at higher production quartiles. The near-parallel slopes are
consistent with the insignificant interaction term found across all regression
specifications.


In [9]:
plot_df = rdf[['Country Code', 'Country Name', 'Year', ECI_COL,
              'log_HCI', 'log_Production_Value', 'ClusterLabels']].dropna().copy()

plot_df['prod_q'] = pd.qcut(plot_df['log_Production_Value'], 4,
                            labels=['Q1 (low NR)', 'Q2', 'Q3', 'Q4 (high NR)'])
PROD_COLORS = {'Q1 (low NR)': '#a8c5da', 'Q2': '#4a6fa5',
               'Q3': '#d4853b', 'Q4 (high NR)': '#c23a3a'}

fig_b5 = go.Figure()
for q, col in PROD_COLORS.items():
    sub = plot_df[plot_df['prod_q'] == q]
    fig_b5.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub[ECI_COL], mode='markers',
        marker=dict(size=5, color=col, opacity=0.65, line=dict(color='white', width=0.5)),
        name=q,
        customdata=np.stack([sub['Country Code'], sub['Country Name'],
                             sub['Year'], sub['log_Production_Value']], axis=1),
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]}, %{customdata[2]})<br>'
                      'log HCI: %{x:.3f}<br>ECI: %{y:.3f}<br>'
                      'log Prod: %{customdata[3]:.3f}<extra>' + q + '</extra>',
    ))

for q, col in PROD_COLORS.items():
    sub = plot_df[plot_df['prod_q'] == q].copy()
    if len(sub) < 10: continue
    xv = sub['log_HCI'].values
    yv = sub[ECI_COL].values
    c  = np.polyfit(xv, yv, 1)
    xl = np.linspace(xv.min(), xv.max(), 80)
    fig_b5.add_trace(go.Scatter(
        x=xl, y=np.polyval(c, xl), mode='lines',
        line=dict(color=col, width=2, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))

fig_b5.update_layout(**base_layout(
    height=560,
    margin=dict(l=70, r=40, t=55, b=60),
    xaxis=dict(title='Human Capital Index (log)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(title='NR Production quartile', font=dict(size=11),
                bgcolor='rgba(255,255,255,0.9)', bordercolor=GRID, borderwidth=1),
))
save(fig_b5, 'B5_reg_hci_production_interaction', w=1100, h=560)
fig_b5.show(config=CFG)


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B5_reg_hci_production_interaction.html


  Saved: /Users/leoss/Desktop/GitHub/Capstone/CLEAN/Visualisation/charts/appendix/B5_reg_hci_production_interaction.png


## Done

All five appendix charts saved to `Visualisation/charts/appendix/`.

| Chart | File | Content |
|-------|------|---------|
| B1 | `B1_cluster_map_k5_1995` | 47-country cluster world map |
| B2 | `B2_ml_feature_importance_consensus` | LASSO / Ridge / EN importance |
| B3 | `B3_ml_train_vs_test_r2` | ECI level and delta-ECI R-squared |
| B4 | `B4_reg_coefficient_forest_3a_3d` | Regression coefficients across 4 specs |
| B5 | `B5_reg_hci_production_interaction` | HCI x Production scatter by quartile |
